# Final clean version

## YOLO-World-v2-S — LVIS minival Evaluation (Results Reference)

> **⚠️ Note: This notebook cannot be re-run as-is.**
>
> `mmcv==2.0.1` is installed from `download.openmmlab.com`, which has been consistently
> timing out from Google Colab since this evaluation was conducted. Because `mmcv` is a
> hard dependency of `mmdet` and `mmyolo`, the full environment cannot be reproduced
> without a working CDN connection.
>
> **The saved cell outputs below are the actual results from the original run.**

---

**Model:** `yolo_world_v2_s_obj365v1_goldg_pretrain.pth`  
**Dataset:** LVIS minival (4,809 images, 1,203 categories)  
**Environment:** Google Colab, Python 3.10, PyTorch 2.0.1+cu118, mmcv 2.0.1

| Metric | Value |
|--------|-------|
| AP     | 22.70 |
| APr    | 16.30 |
| APc    | 20.80 |
| APf    | 25.50 |

## Own Code
Builds the Colab evaluation environment, patches LVIS annotation compatibility, and runs the official YOLO-World evaluation with log checks.

In [ ]:
# YOLO-World evaluation setup
# Setup steps + np.float patch

# 1) Python 3.10 conda env
!conda create -n yworld_py310 python=3.10 -y

# 2) PyTorch 2.0.1 + CUDA 11.8
!conda run -n yworld_py310 pip install -q torch==2.0.1 torchvision==0.15.2 --index-url https://download.pytorch.org/whl/cu118

# 3) OpenMMLab / YOLO-World dependencies
!conda run -n yworld_py310 pip install -q --force-reinstall "numpy==1.26.4"
!conda run -n yworld_py310 pip install -q "mmengine>=0.7.0,<1.0.0"
!conda run -n yworld_py310 pip install -q "mmcv==2.0.1" -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html
!conda run -n yworld_py310 pip install -q "mmdet==3.0.0"
!conda run -n yworld_py310 pip install -q "mmyolo>=0.6.0"
!conda run -n yworld_py310 pip install -q "transformers==4.36.2" "tokenizers<0.19"
!conda run -n yworld_py310 pip install -q lvis pycocotools supervision==0.19.0
!conda run -n yworld_py310 pip install -q "setuptools<70"

# 4) YOLO-World clone
%cd /content
!rm -rf yolo_world
!git clone https://github.com/AILab-CVC/YOLO-World.git yolo_world
%cd /content/yolo_world

# 5) Git submodules
!git submodule update --init --recursive

# 6) Editable install
!conda run -n yworld_py310 pip install -q -e . --no-deps

# 7) Detector syntax patch
!sed -i 's/self.text_feats, None = self.backbone.forward_text(texts)/self.text_feats, _ = self.backbone.forward_text(texts)/' /content/yolo_world/yolo_world/models/detectors/yolo_world.py

# 8) LVIS NumPy patch
!sed -i.bak 's/np\.float/float/g' /usr/local/envs/yworld_py310/lib/python3.10/site-packages/lvis/eval.py

# 9) Import checks
!MPLBACKEND=Agg conda run -n yworld_py310 python -c "import torch; print('torch', torch.__version__, 'cuda', torch.cuda.is_available())"
!MPLBACKEND=Agg conda run -n yworld_py310 python -c "import numpy; print('numpy', numpy.__version__)"
!MPLBACKEND=Agg conda run -n yworld_py310 python -c "import mmcv; print('mmcv', mmcv.__version__); import mmcv._ext; print('mmcv ext ok')"
!MPLBACKEND=Agg conda run -n yworld_py310 python -c "import mmdet; print('mmdet ok')"
!MPLBACKEND=Agg conda run -n yworld_py310 python -c "import mmyolo; print('mmyolo ok')"
!MPLBACKEND=Agg conda run -n yworld_py310 python -c "import transformers; print('transformers ok')"
!MPLBACKEND=Agg conda run -n yworld_py310 python -c "import yolo_world; print('yolo_world ok')"
!grep -n "np.float" /usr/local/envs/yworld_py310/lib/python3.10/site-packages/lvis/eval.py || echo "np.float patch complete"

Channels:
 - conda-forge
Platform: linux-64
Solving environment: / - done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.2
    latest version: 26.5.2

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local/envs/yworld_py310

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    bzip2-1.0.8                |       hda65f42_9         254 KB  conda-forge
    ca-certificates-2026.5.20  |       hbd8a1cb_0         127 KB  conda-forge
    ld_impl_linux-64-2.45.1    |default_hbd61a6d_102         711 KB  conda-forge
    libexpat-2.8.1             |       hecca717_0          75 KB  conda-forge
    libffi-3.5.2               |       h3435931_0          57 KB  conda-fo

In [ ]:
%cd /content/yolo_world

# 1) Download checkpoint
!mkdir -p checkpoints
!wget -q -O checkpoints/yolo_world_v2_s_obj365v1_goldg_pretrain.pth \
  https://huggingface.co/wondervictor/YOLO-World/resolve/main/yolo_world_v2_s_obj365v1_goldg_pretrain.pth

!ls -lh checkpoints/yolo_world_v2_s_obj365v1_goldg_pretrain.pth

# 2) LVIS annotation check
!ls -lh /content/lvis_v1_minival.json || echo "LVIS json not found. Upload lvis_v1_minival.json first."

/content/yolo_world
-rw-r--r-- 1 root root 0 Jun  4 14:32 checkpoints/yolo_world_v2_s_obj365v1_goldg_pretrain.pth
-rw-r--r-- 1 root root 34M Jun  4 14:32 /content/lvis_v1_minival.json


In [ ]:
%cd /content/yolo_world

!rm -f checkpoints/yolo_world_v2_s_obj365v1_goldg_pretrain.pth

!wget -q --show-progress \
  https://huggingface.co/wondervictor/YOLO-World/resolve/main/yolo_world_v2_s_obj365v1_goldg_pretrain-55b943ea.pth \
  -O checkpoints/yolo_world_v2_s_obj365v1_goldg_pretrain.pth

!ls -lh checkpoints/yolo_world_v2_s_obj365v1_goldg_pretrain.pth

/content/yolo_world
checkpoints/yolo_wo 100%[===================>] 290.92M  13.9MB/s    in 35s     
-rw-r--r-- 1 root root 291M Jun  4 14:33 checkpoints/yolo_world_v2_s_obj365v1_goldg_pretrain.pth


In [ ]:
%cd /content/yolo_world

# 1) LVIS annotation directory
!mkdir -p data/coco/lvis

# 2) Add file_name from coco_url
import json, os

src = "/content/lvis_v1_minival.json"
dst = "/content/yolo_world/data/coco/lvis/lvis_v1_minival_inserted_image_name.json"

with open(src, "r") as f:
    data = json.load(f)

for img in data["images"]:
    if "file_name" not in img or not img["file_name"]:
        coco_url = img.get("coco_url", "")
        img["file_name"] = coco_url.split("/")[-1]

with open(dst, "w") as f:
    json.dump(data, f)

print("saved:", dst)
print("images:", len(data["images"]))
print("first:", data["images"][0])

# 3) COCO val2017 images
!mkdir -p data/coco
%cd /content/yolo_world/data/coco

!wget -q --show-progress http://images.cocodataset.org/zips/val2017.zip
!unzip -q val2017.zip
!rm val2017.zip

# 4) Final check
!ls -lh /content/yolo_world/data/coco/lvis/lvis_v1_minival_inserted_image_name.json
!find /content/yolo_world/data/coco/val2017 -iname "*.jpg" | wc -l

/content/yolo_world
saved: /content/yolo_world/data/coco/lvis/lvis_v1_minival_inserted_image_name.json
images: 4809
first: {'date_captured': '2013-11-14 17:02:52', 'neg_category_ids': [279, 899, 127, 180, 1136, 725, 663], 'id': 397133, 'license': 4, 'height': 427, 'width': 640, 'flickr_url': 'http://farm7.staticflickr.com/6116/6255196340_da26cf2c9e_z.jpg', 'coco_url': 'http://images.cocodataset.org/val2017/000000397133.jpg', 'not_exhaustive_category_ids': [914, 801, 566, 139, 1021], 'file_name': 'val2017/000000397133.jpg'}
/content/yolo_world/data/coco
val2017.zip         100%[===================>] 777.80M  10.7MB/s    in 60s     
-rw-r--r-- 1 root root 34M Jun  4 14:34 /content/yolo_world/data/coco/lvis/lvis_v1_minival_inserted_image_name.json
5000


In [ ]:
%cd /content/yolo_world

# 1) np.float patch check
!grep -n "np.float" /usr/local/envs/yworld_py310/lib/python3.10/site-packages/lvis/eval.py || echo "np.float patch complete"

# 2) Create output directory
!mkdir -p work_dirs/yworldv2s_lvis_minival_eval_final

# 3) LVIS minival eval
!nohup env MPLBACKEND=Agg conda run --no-capture-output -n yworld_py310 python tools/test.py \
  configs/pretrain/yolo_world_v2_s_vlpan_bn_2e-3_100e_4x8gpus_obj365v1_goldg_train_lvis_minival.py \
  checkpoints/yolo_world_v2_s_obj365v1_goldg_pretrain.pth \
  --work-dir work_dirs/yworldv2s_lvis_minival_eval_final \
  > work_dirs/yworldv2s_lvis_minival_eval_final/run.log 2>&1 &

# 4) Startup log
!sleep 5
!tail -n 40 work_dirs/yworldv2s_lvis_minival_eval_final/run.log

/content/yolo_world
np.float patch complete
06/04 14:42:57 - mmengine - WARNING - Failed to search registry with scope "mmyolo" in the "log_processor" registry tree. As a workaround, the current "log_processor" registry in "mmengine" is used to build instance. This may cause unexpected failure when running the built modules. Please check whether "mmyolo" is a correct scope, or whether the registry is initialized.


In [ ]:
# Evaluation results
!tail -n 40 /content/yolo_world/work_dirs/yworldv2s_lvis_minival_eval_final/run.log

# NumPy version issue

06/04 15:31:30 - mmengine - INFO - Epoch(test) [3600/4809]    eta: 0:16:11  time: 0.8150  data_time: 0.0009  memory: 828  
06/04 15:32:11 - mmengine - INFO - Epoch(test) [3650/4809]    eta: 0:15:31  time: 0.8016  data_time: 0.0008  memory: 828  
06/04 15:32:51 - mmengine - INFO - Epoch(test) [3700/4809]    eta: 0:14:51  time: 0.8075  data_time: 0.0008  memory: 828  
06/04 15:33:29 - mmengine - INFO - Epoch(test) [3750/4809]    eta: 0:14:10  time: 0.7691  data_time: 0.0008  memory: 828  
06/04 15:34:11 - mmengine - INFO - Epoch(test) [3800/4809]    eta: 0:13:31  time: 0.8317  data_time: 0.0008  memory: 828  
06/04 15:34:49 - mmengine - INFO - Epoch(test) [3850/4809]    eta: 0:12:50  time: 0.7564  data_time: 0.0009  memory: 828  
06/04 15:35:27 - mmengine - INFO - Epoch(test) [3900/4809]    eta: 0:12:09  time: 0.7673  data_time: 0.0008  memory: 828  
06/04 15:36:05 - mmengine - INFO - Epoch(test) [3950/4809]    eta: 0:11:29  time: 0.7583  data_time: 0.0007  memory: 828  
06/04 15:36:45 -

In [ ]:
# Reinstall numpy
!conda run -n yworld_py310 pip uninstall -y numpy
!conda run -n yworld_py310 pip install --no-cache-dir --force-reinstall "numpy==1.24.4"

Found existing installation: numpy 2.2.6
Uninstalling numpy-2.2.6:
  Successfully uninstalled numpy-2.2.6

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 288.2 MB/s  0:00:00

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
yolo-world 0.1.0 requires mmcv-lite>=2.0.0rc4, which is not installed.
yolo-world 0.1.0 requires openmim, which is not installed.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.24.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.24.4 which is incompatible.
yolo-world 0.1.0 requires torchvision>=0.16.2, but you have torchvision 0.15.2+cu118 which is incompatible.



In [ ]:
# Numpy + torch check
!conda run -n yworld_py310 python -c "import numpy as np; print('numpy', np.__version__)"
!conda run -n yworld_py310 python -c "import torch, numpy as np; print('torch', torch.__version__); print(torch.from_numpy(np.array([1,2,3])))"

# Check passed

numpy 1.24.4

torch 2.0.1+cu118
tensor([1, 2, 3])



In [ ]:
# np.float patch check
!grep -n "np.float" /usr/local/envs/yworld_py310/lib/python3.10/site-packages/lvis/eval.py || echo "np.float patch complete"

# Patch confirmed

np.float patch complete


# Trial runs

Optional details

In [ ]:
!pwd
!ls -lh /content
!python --version
!nvidia-smi

/content
total 4.0K
drwxr-xr-x 1 root root 4.0K May 26 13:31 sample_data
Python 3.12.13
Thu Jun  4 14:16:27 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   44C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |         

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install()

⏬ Downloading https://github.com/jaimergp/miniforge/releases/download/24.11.2-1_colab/Miniforge3-colab-24.11.2-1_colab-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:10
🔁 Restarting kernel...


In [ ]:
!which conda
!conda --version
!python --version

/usr/local/bin/conda
conda 24.11.2
Python 3.11.11


In [ ]:
!conda create -n yworld_py310 python=3.10 -y

Channels:
 - conda-forge
Platform: linux-64
Solving environment: \ | done


==> WARNING: A newer version of conda exists. <==
    current version: 24.11.2
    latest version: 26.5.2

Please update conda by running

    $ conda update -n base -c conda-forge conda



## Package Plan ##

  environment location: /usr/local/envs/yworld_py310

  added / updated specs:
    - python=3.10


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-4.5          |           20_gnu          28 KB  conda-forge
    bzip2-1.0.8                |       hda65f42_9         254 KB  conda-forge
    ca-certificates-2026.5.20  |       hbd8a1cb_0         127 KB  conda-forge
    ld_impl_linux-64-2.45.1    |default_hbd61a6d_102         711 KB  conda-forge
    libexpat-2.8.1             |       hecca717_0          75 KB  conda-forge
    libffi-3.5.2               |       h3435931_0          57 KB  conda-fo

In [ ]:
!conda run -n yworld_py310 python --version

Python 3.10.20



In [ ]:
# Install PyTorch

!conda run -n yworld_py310 pip install -q torch==2.0.1 torchvision==0.15.2 --index-url https://download.pytorch.org/whl/cu118

In [ ]:
# Verify PyTorch

!conda run -n yworld_py310 python - << 'PY'
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)

/bin/bash: line 1: warning: here-document at line 1 delimited by end-of-file (wanted `PY')
torch: 2.11.0+cu128
cuda available: True
gpu: Tesla T4


In [ ]:
# Version mismatch check
# Env path check

!conda run -n yworld_py310 which python
!conda run -n yworld_py310 which pip
!conda run -n yworld_py310 pip show torch | head -n 20

# Env version is correct.

/usr/local/envs/yworld_py310/bin/python

/usr/local/envs/yworld_py310/bin/pip

Name: torch
Version: 2.0.1+cu118
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org/
Author: PyTorch Team
Author-email: packages@pytorch.org
License: BSD-3
Location: /usr/local/envs/yworld_py310/lib/python3.10/site-packages
Requires: filelock, jinja2, networkx, sympy, triton, typing-extensions
Required-by: torchvision, triton



In [ ]:
# Pinned OpenMMLab install

!conda run -n yworld_py310 pip install -q "mmengine>=0.7.0,<1.0.0"
!conda run -n yworld_py310 pip install -q "mmcv==2.0.1" -f https://download.openmmlab.com/mmcv/dist/cu118/torch2.0/index.html
!conda run -n yworld_py310 pip install -q "mmdet>=3.0.0,<3.4.0"
!conda run -n yworld_py310 pip install -q "mmyolo>=0.6.0"
!conda run -n yworld_py310 pip install -q lvis pycocotools

In [ ]:
# OpenMMLab check

!conda run -n yworld_py310 python -c "import mmcv; print('mmcv', mmcv.__version__); import mmcv._ext; print('mmcv ext ok')"
!conda run -n yworld_py310 python -c "import mmdet; print('mmdet ok')"
!conda run -n yworld_py310 python -c "import mmyolo; print('mmyolo ok')"

# Import check passed.

mmcv 2.0.1
mmcv ext ok

mmdet ok

mmyolo ok



In [ ]:
# YOLO-World import check

%cd /content
!rm -rf yolo_world
!git clone https://github.com/AILab-CVC/YOLO-World.git yolo_world
%cd /content/yolo_world

!conda run -n yworld_py310 pip install -q -e . --no-deps
!conda run -n yworld_py310 python -c "import yolo_world; print('yolo_world ok')"

# Matplotlib backend issue
# Colab default backend: matplotlib_inline
# Conda env cannot use this backend
# Use Agg backend

/content
Cloning into 'yolo_world'...
remote: Enumerating objects: 1213, done.
remote: Counting objects: 100% (650/650), done.
remote: Compressing objects: 100% (178/178), done.
remote: Total 1213 (delta 492), reused 472 (delta 472), pack-reused 563 (from 1)
Receiving objects: 100% (1213/1213), 3.95 MiB | 10.08 MiB/s, done.
Resolving deltas: 100% (696/696), done.
/content/yolo_world

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<string>", line 1, in <module>
  File "/content/yolo_world/yolo_world/__init__.py", line 10, in <module>
    from .models import *  # 

In [ ]:
# Agg backend

!MPLBACKEND=Agg conda run -n yworld_py310 python -c "import yolo_world; print('yolo_world ok')"

# Backend issue fixed
# Remaining issues: NumPy 2.x, transformers
# NumPy 2.x -> numpy<2
# Missing transformers


A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "<string>", line 1, in <module>
  File "/content/yolo_world/yolo_world/__init__.py", line 10, in <module>
    from .models import *  # noqa
  File "/content/yolo_world/yolo_world/models/__init__.py", line 2, in <module>
    from .backbones import *  # noqa
  File "/content/yolo_world/yolo_world/models/backbones/__init__.py", line 5, in <module>
    from .mm_backbone import (
  File "/content/yolo_world/yolo_world/models/backbones/mm_backbone.py", line 7, in <module>
    from mmengine.model import BaseModule
  File "

In [ ]:
# numpy downgrade + transformers

!conda run -n yworld_py310 pip install -q "numpy<2" transformers

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
yolo-world 0.1.0 requires mmcv-lite>=2.0.0rc4, which is not installed.
yolo-world 0.1.0 requires openmim, which is not installed.
yolo-world 0.1.0 requires supervision==0.19.0, which is not installed.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
yolo-world 0.1.0 requires mmdet==3.0.0, but you have mmdet 3.3.0 which is incompatible.
yolo-world 0.1.0 requires torchvision>=0.16.2, but you have torchvision 0.15.2+cu118 which is incompatible.



In [ ]:
# Import check
# If mmdet fails: use mmdet==3.0.0

!MPLBACKEND=Agg conda run -n yworld_py310 python -c "import numpy; print('numpy', numpy.__version__)"
!MPLBACKEND=Agg conda run -n yworld_py310 python -c "import transformers; print('transformers ok')"
!MPLBACKEND=Agg conda run -n yworld_py310 python -c "import yolo_world; print('yolo_world ok')"

# Transformers version too high

numpy 1.26.4

transformers ok

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.0.1+cu118
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.0.1+cu118
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/content/yolo_world/yolo_world/__init__.py", line 10, in <module>
    from .models import *  # noqa
  File "/content/yolo_world/yolo_world/models/__init__.py", line 3, in <module>
    from .layers import *  # noqa
  File "/content/yolo_world/yolo_world/models/layers/__init__.py", line 4, in <module>
    from .yolo_bricks import (
  File "/content/yolo_world/yolo_world/models/layers/yolo_bricks.py", line 12, in <module>
    from mm

In [ ]:
# Transformers downgrade
!conda run -n yworld_py310 pip install -q "transformers==4.36.2" "tokenizers<0.19"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
yolo-world 0.1.0 requires mmcv-lite>=2.0.0rc4, which is not installed.
yolo-world 0.1.0 requires openmim, which is not installed.
yolo-world 0.1.0 requires supervision==0.19.0, which is not installed.
yolo-world 0.1.0 requires mmdet==3.0.0, but you have mmdet 3.3.0 which is incompatible.
yolo-world 0.1.0 requires torchvision>=0.16.2, but you have torchvision 0.15.2+cu118 which is incompatible.



In [ ]:
!MPLBACKEND=Agg conda run -n yworld_py310 python -c "import torch; print('torch', torch.__version__)"
!MPLBACKEND=Agg conda run -n yworld_py310 python -c "import transformers; print('transformers', transformers.__version__)"
!MPLBACKEND=Agg conda run -n yworld_py310 python -c "import yolo_world; print('yolo_world ok')"

torch 2.0.1+cu118

transformers 4.36.2

Traceback (most recent call last):
  File "<string>", line 1, in <module>
  File "/content/yolo_world/yolo_world/__init__.py", line 10, in <module>
    from .models import *  # noqa
  File "/content/yolo_world/yolo_world/models/__init__.py", line 4, in <module>
    from .detectors import *  # noqa
  File "/content/yolo_world/yolo_world/models/detectors/__init__.py", line 2, in <module>
    from .yolo_world import YOLOWorldDetector, SimpleYOLOWorldDetector
  File "/content/yolo_world/yolo_world/models/detectors/yolo_world.py", line 61
    self.text_feats, None = self.backbone.forward_text(texts)
                     ^^^^
SyntaxError: cannot assign to None

ERROR conda.cli.main_run:execute(125): `conda run python -c import yolo_world; print('yolo_world ok')` failed. (See above for error)
